# Real Image Editing — One-Shot Colab Notebook

End-to-end run on **one image + one instruction**, producing two paths and three visualizations.

**Method path** (PEZ + P2P):
  1. PEZ-1 inversion → continuous source embeddings + per-timestep null-text
  2. PEZ-2 instruction-conditioned target prompt sweep over Knob 1 (`λ_instruction`, `γ_anchor`)
  3. P2P + LocalBlend edit sweep over Knob 2 (`cross_replace_steps`)

**Baseline path** (BLIP + manual prompt):
  1. BLIP captions the source
  2. User-supplied target prompt (set after seeing the caption)
  3. Standard P2P + LocalBlend edit over Knob 2

**Visualizations:**
  1. Prompt evolution PEZ-1 → PEZ-2 across Knob 1
  2. 2D matrix: rows = Knob 1 (PEZ-2 prompts), cols = Knob 2 (cross_replace_steps)
  3. 1D row: BLIP-source + manual-target P2P, cols = same Knob 2 axis as #2

**Outputs saved to `outputs/colab_oneshot/`:** source.png, every grid cell as an individual PNG, the composed Vis #2 / Vis #3 figures, the Vis #1 prompt-diff HTML, and a `run_metadata.json` with all prompts, knob settings, BLIP caption, and model IDs.

**Recommended run order:** the BLIP-caption cell runs in seconds and prints the caption *before* PEZ-1 (~70 min) starts, so you can write a good `MANUAL_TARGET_PROMPT` while looking at the caption.

**Expected runtime on A100:** ~90–150 min cold (PEZ-1 dominates). PEZ-1 and PEZ-2 results are cached, so re-runs with the same image/instruction are fast.

**GPU:** A100 (40 GB) recommended. Cold weights total ~10 GB (SD2.1 + OpenCLIP-H/14 + BLIP); T4 (15 GB) may OOM during PEZ-1 backward.

**Requires:** `HF_TOKEN` Colab secret (key icon in left sidebar) for SD2.1 weights.

In [ ]:
# 1. Clone repo and install deps not already on Colab
import os
if not os.path.isdir('real-image-editing'):
    !git clone https://github.com/beratcelik1/real-image-editing.git
%cd real-image-editing
!git submodule update --init --recursive 2>/dev/null
!pip install -q diffusers transformers accelerate safetensors Pillow scikit-image matplotlib pandas

import torch
assert torch.cuda.is_available(), 'GPU runtime required (Runtime → Change runtime type → GPU)'
print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. HuggingFace login
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via Colab secret')
except Exception:
    if os.environ.get('HF_TOKEN'):
        login(token=os.environ['HF_TOKEN'])
        print('Logged in via env var')
    else:
        print('WARNING: No HF_TOKEN found. SD2.1 download may fail.')

In [ ]:
# 3. Inputs — change these for your run.
#    The MANUAL_TARGET_PROMPT for the BLIP baseline lives in its own cell
#    further down so you can set it AFTER seeing the BLIP caption.

# 3a. Source image. Default: the local husky+chihuahua example
#     (cropped to 512x512 in data/examples/). To use a different image,
#     either set IMAGE_PATH to another local file or set IMAGE_URL and
#     IMAGE_PATH together (auto-fetch fallback below).
IMAGE_PATH = 'data/examples/huskychihuahua_512.png'
IMAGE_URL = ''   # blank when using a pre-prepped local file
if IMAGE_URL and not os.path.isfile(IMAGE_PATH):
    os.makedirs(os.path.dirname(IMAGE_PATH), exist_ok=True)
    !wget -q -O {IMAGE_PATH} "{IMAGE_URL}"

# 3b. Method-path edit instruction (used by PEZ-2)
INSTRUCTION = 'cats on a couch'   # rich description (proposal §8a):
                                  # the cosine-cancellation property protects
                                  # 'couch' content already in PEZ-1's prompt;
                                  # drift concentrates on the husky/chihuahua
                                  # → cat axis. Multi-subject test for §3.2.

# 3c. Knob 1 grids — PEZ-2 hyperparameters. The full (λ, γ) sweep
#     defined in RESEARCH_PROPOSAL.md §4.5. Visualizations use λ as the
#     horizontal axis and γ as the vertical axis.
LAMBDA_GRID = [0.5, 1.0, 2.0]           # λ_instruction (3×3 sweep — drops 5.0,
                                        # the most aggressive end where alignment
                                        # tends to break; cf. RESEARCH_PROPOSAL.md §4.2)
GAMMA_GRID  = [0.2, 0.02, 0.002]        # γ_anchor (re-centered for N=75: at higher
                                        # N, L_anchor's total grows — re-scale γ ~1/5
                                        # vs N=15 defaults to keep effective per-position
                                        # anchor balance similar; sweep covers ~2 decades)

# 3d. Knob 2 grid — cross_replace_steps. Secondary axis: rendered as
#     three (λ × γ) panels (one per cross_replace_steps value), GIF
#     frames, and strip-comparison columns.
KNOB_2_GRID = [0.8, 0.5, 0.3]

# Per-(image, instruction) cost (3×3 grid): 9 PEZ-2 runs +
# 27 edits ≈ 1-1.5 h with adaptive early stopping in pez_search.
# Cost-aware ordering (PEZ-2 outer, editing inner) keeps PEZ-2
# amortized across cross_replace_steps values.

# 3e. Output directory (everything saved here for offline review)
from pathlib import Path
OUTPUT_DIR = Path('outputs/colab_oneshot')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from PIL import Image
Image.open(IMAGE_PATH).resize((256, 256))

In [ ]:
# 4. Load models: SD2.1 + CLIP (for PEZ-1 image features) + BLIP (for caption baseline)
#
# CLIP-variant choice: SD2.1-base's text encoder is OpenCLIP-H/14 (hidden_size=1024).
# The CLIP loaded here MUST be 1024-dim or `clip_similarity_loss` errors on the
# cosine-similarity dot product (SD-pooled vs. clip_image_features). We use the
# HF-compatible OpenCLIP-H/14 checkpoint from LAION.
from src.utils import (
    load_sd_components, load_image, encode_image, decode_latent,
    get_text_embeddings, get_uncond_embeddings, get_device,
)
from src.config import load_pez_1, load_pez_2, load_local_blend, load_edit

device = get_device()

edit_config = load_edit()
print(f'Loading {edit_config.sd_model}...')
unet, vae, text_encoder, tokenizer, scheduler = load_sd_components(
    model_id=edit_config.sd_model,
)
sd_components = {
    'unet': unet, 'vae': vae, 'text_encoder': text_encoder,
    'tokenizer': tokenizer, 'scheduler': scheduler,
}

# OpenCLIP-H/14 (HF format) — matches SD2.1-base's 1024-dim text encoder.
from transformers import CLIPModel, CLIPProcessor
CLIP_HF_ID = 'laion/CLIP-ViT-H-14-laion2B-s32B-b79K'
clip_model = CLIPModel.from_pretrained(CLIP_HF_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_HF_ID)
text_projection = clip_model.text_projection  # nn.Linear(1024, 1024)

# BLIP for the baseline caption
from transformers import BlipForConditionalGeneration, BlipProcessor
BLIP_HF_ID = 'Salesforce/blip-image-captioning-large'
blip_processor = BlipProcessor.from_pretrained(BLIP_HF_ID)
blip_model = BlipForConditionalGeneration.from_pretrained(BLIP_HF_ID).to(device).eval()

source_image = load_image(IMAGE_PATH, size=512)
with torch.no_grad():
    inputs = clip_processor(images=source_image, return_tensors='pt').to(device)
    clip_image_features = clip_model.get_image_features(**inputs)

# Free the rest of the CLIP vision/text weights from VRAM. text_projection was
# captured above as a separate reference and stays alive on device.
del clip_model
torch.cuda.empty_cache()
print('Models loaded.')

In [ ]:
# 5. BLIP caption — runs in seconds. Look at the printed caption, then
#    set MANUAL_TARGET_PROMPT in the next cell. Doing this BEFORE PEZ-1
#    means you decide the baseline target while PEZ-1 is still queued.
with torch.no_grad():
    inputs = blip_processor(images=source_image, return_tensors='pt').to(device)
    out = blip_model.generate(**inputs, max_length=30)
BLIP_SOURCE_PROMPT = blip_processor.decode(out[0], skip_special_tokens=True)
print(f'BLIP caption: {BLIP_SOURCE_PROMPT!r}')
print('\n→ Edit MANUAL_TARGET_PROMPT in the next cell based on this caption.')

# Free BLIP weights — we already have the caption string.
del blip_model, blip_processor
torch.cuda.empty_cache()

In [ ]:
# 6. Manual target prompt for the BLIP baseline path. Edit this AFTER
#    seeing the caption above. The baseline runs `BLIP_SOURCE_PROMPT →
#    MANUAL_TARGET_PROMPT` through standard P2P (no PEZ).
MANUAL_TARGET_PROMPT = 'two cats on a couch'   # BLIP-baseline target;
                                                # mirror the method-path INSTRUCTION
print(f'Source (from BLIP): {BLIP_SOURCE_PROMPT!r}')
print(f'Target (manual):    {MANUAL_TARGET_PROMPT!r}')

---
## Phase A — Method path (PEZ-1 → PEZ-2 → P2P)

In [ ]:
# 7. PEZ-1: continuous source-prompt inversion (cached on disk)
from src.pez.source_inversion import pez_invert_source
from src.pez.search import nn_project

print('Running PEZ-1 (~70 min on first run, instant on cache hit)...')
pez_1_config = load_pez_1()
source_embeddings, null_text_per_timestep = pez_invert_source(
    image=source_image,
    config=pez_1_config,
    sd_components=sd_components,
    clip_image_features=clip_image_features,
    text_projection=text_projection,
)
# source_embeddings: Tensor[1, N, 768] — the canonical artifact.
# Project to nearest vocab for human-readable logging only:
_, _projected_ids = nn_project(
    source_embeddings,
    sd_components['text_encoder'].text_model.embeddings.token_embedding,
)
pez_1_prompt = tokenizer.decode(_projected_ids[0].tolist(), skip_special_tokens=True)
print(f'PEZ-1 (projected for inspection): {pez_1_prompt!r}')
print(f'PEZ-1 embeddings shape: {tuple(source_embeddings.shape)}')

In [ ]:
# 8. PEZ-2 sweep over the full (λ, γ) grid (cached per-setting)
from dataclasses import replace
from src.pez.instruction_conditioned import pez_invert_with_instruction
from src.pez.search import nn_project

pez_2_config = load_pez_2()
pez_2_results = {}  # (λ, γ) -> (target_embeddings, decoded_prompt)
_token_emb = sd_components['text_encoder'].text_model.embeddings.token_embedding

for lam in LAMBDA_GRID:
    for gam in GAMMA_GRID:
        print(f'\n[PEZ-2] λ={lam}, γ={gam}')
        cfg = replace(pez_2_config, lambda_instruction=lam, gamma_anchor=gam)
        target_embeddings = pez_invert_with_instruction(
            image=source_image,
            instruction=INSTRUCTION,
            pez_1_embeddings=source_embeddings,
            null_text_per_timestep=null_text_per_timestep,
            config=cfg,
            sd_components=sd_components,
        )
        # Project to nearest vocab for human-readable logging:
        _, _projected = nn_project(target_embeddings, _token_emb)
        target_prompt = tokenizer.decode(_projected[0].tolist(), skip_special_tokens=True)
        pez_2_results[(lam, gam)] = (target_embeddings, target_prompt)
        print(f'  → {target_prompt!r}')

print(f'\nGenerated {len(pez_2_results)} PEZ-2 outputs.')

In [ ]:
# 9. Visualization 1 — PEZ-1 → PEZ-2 evolution across the full (λ, γ) grid
#
# Three views, in increasing fidelity to what continuous-PEZ encodes:
#   (a) (λ × γ) 2D drift summary heatmap — single scalar (mean cosine
#       to PEZ-1) per (λ, γ) cell. The (λ, γ)-centric overview.
#   (b) Per-position cosine heatmap — 12 rows × N cols, the detail view.
#   (c) Top-3 nearest vocab per position — surfaces between-vocab
#       embeddings.
import base64
import difflib
import io

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display, HTML

_token_emb = sd_components['text_encoder'].text_model.embeddings.token_embedding
src_emb_flat = source_embeddings.squeeze(0).float()              # [N, D]
N = src_emb_flat.shape[0]

# --- (a) (λ × γ) drift summary heatmap ---
# One scalar per (λ, γ): mean cosine similarity to PEZ-1 across all
# positions. Higher mean = embedding stayed close to PEZ-1 overall.
mean_cos = np.zeros((len(GAMMA_GRID), len(LAMBDA_GRID)))
for i, gam in enumerate(GAMMA_GRID):
    for j, lam in enumerate(LAMBDA_GRID):
        tgt_emb, _ = pez_2_results[(lam, gam)]
        sims = F.cosine_similarity(src_emb_flat, tgt_emb.squeeze(0).float(), dim=-1)
        mean_cos[i, j] = sims.mean().item()

fig_a, ax_a = plt.subplots(figsize=(0.9 * len(LAMBDA_GRID) + 2, 0.9 * len(GAMMA_GRID) + 2))
im_a = ax_a.imshow(mean_cos, aspect='auto', cmap='RdYlGn', vmin=0.6, vmax=1.0)
ax_a.set_xticks(range(len(LAMBDA_GRID)))
ax_a.set_xticklabels([f'λ={l}' for l in LAMBDA_GRID])
ax_a.set_yticks(range(len(GAMMA_GRID)))
ax_a.set_yticklabels([f'γ={g}' for g in GAMMA_GRID])
ax_a.set_title('(a) (λ × γ) drift summary\nMean cos_sim(PEZ-1, PEZ-2) per setting')
for i in range(len(GAMMA_GRID)):
    for j in range(len(LAMBDA_GRID)):
        c = 'white' if mean_cos[i, j] < 0.78 else 'black'
        ax_a.text(j, i, f'{mean_cos[i, j]:.3f}', ha='center', va='center', color=c, fontsize=9)
fig_a.colorbar(im_a, ax=ax_a, label='mean cos_sim')
fig_a.tight_layout()
fig_a.savefig(OUTPUT_DIR / 'vis1a_lambda_gamma_drift_summary.png', dpi=120, bbox_inches='tight')

buf_a = io.BytesIO()
fig_a.savefig(buf_a, format='png', dpi=120, bbox_inches='tight')
buf_a.seek(0)
heatmap_a_b64 = base64.b64encode(buf_a.read()).decode('ascii')
plt.show()

# --- (b) Per-position cosine heatmap (12 rows × N positions) ---
settings = [(lam, gam) for lam in LAMBDA_GRID for gam in GAMMA_GRID]
labels   = [f'λ={lam}, γ={gam}' for (lam, gam) in settings]

cos_matrix = []
for (lam, gam) in settings:
    tgt_emb, _ = pez_2_results[(lam, gam)]
    cos_matrix.append(
        F.cosine_similarity(src_emb_flat, tgt_emb.squeeze(0).float(), dim=-1).cpu().tolist()
    )
cos_arr = np.array(cos_matrix)
tau = edit_config.alignment_threshold

fig_b, ax_b = plt.subplots(figsize=(max(7.0, 0.55 * N + 2.0), 0.45 * len(settings) + 2.0))
im_b = ax_b.imshow(cos_arr, aspect='auto', cmap='RdYlGn', vmin=0.6, vmax=1.0)
ax_b.set_xticks(range(N))
ax_b.set_xticklabels(range(N))
ax_b.set_yticks(range(len(settings)))
ax_b.set_yticklabels(labels, fontsize=9)
ax_b.set_xlabel('Position')
ax_b.set_title(f'(b) Per-position cosine to PEZ-1\n'
               f'low = drifted (unmapped at τ={tau:.2f}); high = stable (matched)')
for i in range(len(settings)):
    for j in range(N):
        v = cos_arr[i, j]
        c = 'white' if v < 0.78 else 'black'
        weight = 'bold' if v < tau else 'normal'
        ax_b.text(j, i, f'{v:.2f}', ha='center', va='center',
                  color=c, fontsize=7, fontweight=weight)
fig_b.colorbar(im_b, ax=ax_b, label='cos_sim')
fig_b.tight_layout()
fig_b.savefig(OUTPUT_DIR / 'vis1b_position_drift.png', dpi=120, bbox_inches='tight')

buf_b = io.BytesIO()
fig_b.savefig(buf_b, format='png', dpi=120, bbox_inches='tight')
buf_b.seek(0)
heatmap_b_b64 = base64.b64encode(buf_b.read()).decode('ascii')
plt.show()

# --- (c) Top-3 nearest vocab per position ---
def _topk_nearest(emb, k=3):
    e = emb.squeeze(0).float()
    e_n = F.normalize(e, dim=-1)
    vocab_n = F.normalize(_token_emb.weight.float(), dim=-1)
    sims = e_n @ vocab_n.T
    vals, ids = sims.topk(k, dim=-1)
    return [
        [
            (tokenizer.decode([int(tid)], skip_special_tokens=True).strip() or '<sp>',
             float(v))
            for tid, v in zip(ids[pos].tolist(), vals[pos].tolist())
        ]
        for pos in range(e.shape[0])
    ]

topk_pez1 = _topk_nearest(source_embeddings)
topk_pez2 = {(lam, gam): _topk_nearest(emb) for (lam, gam), (emb, _) in pez_2_results.items()}

def _fmt_topk(entries):
    return ', '.join(f'{tok!r} ({val:.2f})' for tok, val in entries)

tk_rows = []
for pos in range(N):
    row = {'pos': pos, 'PEZ-1': _fmt_topk(topk_pez1[pos])}
    for (lam, gam) in settings:
        row[f'λ={lam}, γ={gam}'] = _fmt_topk(topk_pez2[(lam, gam)][pos])
    tk_rows.append(row)
topk_df = pd.DataFrame(tk_rows)
display(HTML('<h4>(c) Top-3 nearest vocab per position</h4>'))
display(topk_df)

# --- Projected-prompt table for human inspection ---
proj_rows = [{'Setting': 'PEZ-1 (source)', 'λ': '—', 'γ': '—', 'Projected prompt': pez_1_prompt}]
for (lam, gam) in settings:
    _, prompt = pez_2_results[(lam, gam)]
    proj_rows.append({'Setting': f'PEZ-2 (λ={lam}, γ={gam})', 'λ': lam, 'γ': gam, 'Projected prompt': prompt})
proj_df = pd.DataFrame(proj_rows)
display(HTML('<h4>Projected prompts (nearest-vocab decode for inspection only)</h4>'))
display(proj_df)

# --- Persist Vis #1 — full self-contained HTML ---
vis1_full_html = (
    '<!doctype html><meta charset="utf-8">'
    '<style>'
    'body{font-family:sans-serif;max-width:1400px;margin:2em auto;padding:0 1em}'
    'table{border-collapse:collapse;margin:1em 0;font-size:.78em}'
    'td,th{padding:.3em .5em;border:1px solid #ddd;vertical-align:top}'
    'img{max-width:100%}'
    '</style>'
    f'<h3>Vis #1 — PEZ-1 → PEZ-2 evolution across (λ, γ) grid</h3>'
    f'<p><b>Instruction:</b> {INSTRUCTION!r}</p>'
    f'<p><b>PEZ-1 (projected):</b> {pez_1_prompt}</p>'
    f'<h4>(a) (λ × γ) drift summary heatmap</h4>'
    f'<img src="data:image/png;base64,{heatmap_a_b64}">'
    f'<h4>(b) Per-position cosine heatmap</h4>'
    f'<img src="data:image/png;base64,{heatmap_b_b64}">'
    f'<h4>Projected-prompt table</h4>{proj_df.to_html(index=False)}'
    f'<h4>(c) Top-3 nearest vocab per position</h4>'
    f'{topk_df.to_html(index=False, escape=True)}'
)
(OUTPUT_DIR / 'vis1_prompt_evolution.html').write_text(vis1_full_html, encoding='utf-8')
print(f"Saved Vis #1 → {OUTPUT_DIR / 'vis1_prompt_evolution.html'}")
print(f"Saved Vis #1a → {OUTPUT_DIR / 'vis1a_lambda_gamma_drift_summary.png'}")
print(f"Saved Vis #1b → {OUTPUT_DIR / 'vis1b_position_drift.png'}")

In [ ]:
# 10. Helper: thin wrapper around src.pipeline.run_p2p_edit that
#     applies a Knob-2 override to the loaded EditConfig before
#     dispatching. Threads PEZ-1's null_text + a pre-computed z_T so
#     the inner sweep loop avoids redoing inversion 27 times and uses
#     Mokady-style null-text inversion at editing time.
from src.pipeline import run_p2p_edit, prepare_source_inversion, _apply_overrides_nested

# Compute z_T once for the entire sweep — source_embeddings doesn't
# change across (λ, γ, cross_replace_steps) cells. Saves ~26 redundant
# DDIM inversions per (image, instruction) pair (Bug #6 fix).
print('Pre-computing z_T (DDIM inversion of source) for the sweep...')
_base_edit_cfg = load_edit()
cached_z_T = prepare_source_inversion(
    image=source_image,
    source_embeddings=source_embeddings,
    sd_components=sd_components,
    edit_config=_base_edit_cfg,
)
print(f'  z_T shape: {tuple(cached_z_T.shape)}; reusing across all sweep cells')


def run_edit(source_emb, target_emb, cross_replace_steps, *, use_null_text=True):
    """Edit `source_image` from source embeddings to target embeddings at
    the requested cross_replace_steps. Returns the edited PIL image.

    `source_emb` / `target_emb` are continuous [1, N, 768] tensors as
    produced by PEZ-1 / PEZ-2 (or via vocab lookup of token IDs for the
    BLIP baseline path).

    `use_null_text` toggles whether to thread PEZ-1's null_text into
    the editing loop (Bug #2 fix — faithful source reconstruction at
    edit time). Set False for the BLIP baseline since it doesn't have
    PEZ-1 null-text.
    """
    edit_cfg = _apply_overrides_nested(
        _base_edit_cfg,
        {'cross_attention.cross_replace_steps': cross_replace_steps},
    )
    return run_p2p_edit(
        image=source_image,
        source_embeddings=source_emb,
        target_embeddings=target_emb,
        sd_components=sd_components,
        edit_config=edit_cfg,
        local_blend_config=load_local_blend(),
        null_text_per_timestep=null_text_per_timestep if use_null_text else None,
        cached_z_T=cached_z_T if use_null_text else None,
    )

In [ ]:
# 11. Run the full 3D grid: 12 (λ, γ) × 3 cross_replace_steps = 36 edits.
#     Cost-aware order: PEZ-2 outer (already done above), editing inner.
grid_results = {}  # (λ, γ, cross_replace) -> edited_image
for (lam, gam), (target_emb, _) in pez_2_results.items():
    for k2_val in KNOB_2_GRID:
        print(f'Edit [λ={lam}, γ={gam}, cross_replace={k2_val}]')
        grid_results[(lam, gam, k2_val)] = run_edit(
            source_embeddings, target_emb, k2_val,
        )
print(f'\nGenerated {len(grid_results)} method edits.')

In [ ]:
# 12. Visualization 2A — three (λ × γ) image panels, one per
#     cross_replace_steps. (λ, γ)-centric: λ on x-axis, γ on y-axis.
import matplotlib.pyplot as plt

panel_paths = []
for k2_val in KNOB_2_GRID:
    fig, axes = plt.subplots(
        len(GAMMA_GRID), len(LAMBDA_GRID),
        figsize=(2.5 * len(LAMBDA_GRID), 2.5 * len(GAMMA_GRID)),
        squeeze=False,
    )
    for i, gam in enumerate(GAMMA_GRID):
        for j, lam in enumerate(LAMBDA_GRID):
            ax = axes[i, j]
            ax.imshow(grid_results[(lam, gam, k2_val)])
            if i == 0:
                ax.set_title(f'λ={lam}', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'γ={gam}', fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(
        f'Method (PEZ-1 + PEZ-2 + P2P) — instruction: {INSTRUCTION!r}\n'
        f'cross_replace_steps = {k2_val}',
        fontsize=12,
    )
    fig.tight_layout()
    out_path = OUTPUT_DIR / f'vis2a_method_grid_xrep{k2_val}.png'
    fig.savefig(out_path, dpi=120, bbox_inches='tight')
    panel_paths.append(out_path)
    plt.show()
print(f'Saved Vis #2A → {[p.name for p in panel_paths]}')

---
## Phase B — BLIP baseline path (caption + manual target → standard P2P)

In [ ]:
# 13. Tokenize the BLIP source caption + manual target, then look up
#     vocabulary embeddings to feed into run_edit. (BLIP caption was
#     generated way back in cell 5; MANUAL_TARGET_PROMPT was set in cell 6.)
import torch as _torch

_token_emb = sd_components['text_encoder'].text_model.embeddings.token_embedding
_emb_device = _token_emb.weight.device

def _ids_to_embeddings(ids):
    """Look up CLIP vocabulary embeddings for a token-ID list. Returns [1, N, 768]."""
    ids_t = _torch.tensor(ids, device=_emb_device)
    return _token_emb(ids_t).unsqueeze(0).detach()

blip_src_ids = tokenizer.encode(BLIP_SOURCE_PROMPT, add_special_tokens=False)
blip_tgt_ids = tokenizer.encode(MANUAL_TARGET_PROMPT, add_special_tokens=False)

# Pad/truncate to the same length so warm-start-style alignment works.
_n = max(len(blip_src_ids), len(blip_tgt_ids))
def _pad(ids, n):
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    return list(ids) + [pad_id] * (n - len(ids))
blip_src_ids = _pad(blip_src_ids, _n)
blip_tgt_ids = _pad(blip_tgt_ids, _n)

blip_src_embeddings = _ids_to_embeddings(blip_src_ids)
blip_tgt_embeddings = _ids_to_embeddings(blip_tgt_ids)

print(f'Source ({_n} tokens): {BLIP_SOURCE_PROMPT!r}')
print(f'Target ({_n} tokens): {MANUAL_TARGET_PROMPT!r}')

In [ ]:
# 14. 1D sweep — BLIP source + manual target across the same Knob 2
#     axis as vis #2. Pass use_null_text=False since this baseline path
#     doesn't have PEZ-1's null-text, and z_T differs (different source
#     prompt → different inversion).
blip_results = {}
for k2_val in KNOB_2_GRID:
    print(f'BLIP edit [cross_replace={k2_val}]')
    blip_results[k2_val] = run_edit(
        blip_src_embeddings, blip_tgt_embeddings, k2_val,
        use_null_text=False,
    )

In [ ]:
# 15. Visualization 3 — 1D BLIP edit row (shares Knob 2 axis with vis #2).
#     Saves the composed figure as a PNG.
fig, axes = plt.subplots(1, len(KNOB_2_GRID), figsize=(3.5 * len(KNOB_2_GRID), 3.5), squeeze=False)
for j, k2_val in enumerate(KNOB_2_GRID):
    ax = axes[0, j]
    ax.imshow(blip_results[k2_val])
    ax.set_title(f'cross_replace={k2_val}')
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(
    f'Baseline (BLIP source + manual target)\n'
    f'src: {BLIP_SOURCE_PROMPT!r}\n'
    f'tgt: {MANUAL_TARGET_PROMPT!r}',
    fontsize=11,
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'vis3_baseline_row.png', dpi=120, bbox_inches='tight')
print(f"Saved Vis #3 → {OUTPUT_DIR / 'vis3_baseline_row.png'}")
plt.show()

In [ ]:
# 17. Visualization 2B — strip comparison: each (λ, γ) operating point
#     gets a 1×3 row of edits across cross_replace_steps, paired with
#     the BLIP-baseline strip. Mega-figure with 13 rows × 3 cols.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    len(LAMBDA_GRID) * len(GAMMA_GRID) + 1,        # 12 method rows + 1 baseline row
    len(KNOB_2_GRID),                                # 3 cross_replace_steps cols
    figsize=(2.5 * len(KNOB_2_GRID), 2.0 * (len(LAMBDA_GRID) * len(GAMMA_GRID) + 1)),
    squeeze=False,
)

# Header row: cross_replace_steps labels
for j, k2_val in enumerate(KNOB_2_GRID):
    axes[0, j].set_title(f'cross_replace={k2_val}', fontsize=10)

# Method rows: ordered by (λ, γ) — γ outer, λ inner so visually similar
# γs are grouped together (matching the panel layout in Vis #2A).
row = 0
for gam in GAMMA_GRID:
    for lam in LAMBDA_GRID:
        for j, k2_val in enumerate(KNOB_2_GRID):
            ax = axes[row, j]
            ax.imshow(grid_results[(lam, gam, k2_val)])
            ax.set_xticks([]); ax.set_yticks([])
        axes[row, 0].set_ylabel(f'method\nλ={lam}\nγ={gam}', fontsize=9)
        row += 1

# Baseline row (last)
for j, k2_val in enumerate(KNOB_2_GRID):
    ax = axes[row, j]
    ax.imshow(blip_results[k2_val])
    ax.set_xticks([]); ax.set_yticks([])
axes[row, 0].set_ylabel(f'BLIP\nbaseline', fontsize=9, fontweight='bold')

fig.suptitle(
    f'Vis #2B — strip comparison: method (λ, γ) vs BLIP baseline\n'
    f'instruction: {INSTRUCTION!r} | baseline target: {MANUAL_TARGET_PROMPT!r}',
    fontsize=11,
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'vis2b_strip_comparison.png', dpi=110, bbox_inches='tight')
print(f"Saved Vis #2B → {OUTPUT_DIR / 'vis2b_strip_comparison.png'}")
plt.show()

In [ ]:
# 18. Visualization 2C — GIF cycling cross_replace_steps. Each frame
#     is a (γ × λ) method panel + BLIP-baseline sidecar cell, so the
#     reader sees method-vs-baseline at every cross_replace_steps step.
import matplotlib.pyplot as plt
import io

frames = []
target_size = None
for k2_val in KNOB_2_GRID:
    # Layout: (γ rows) × (λ + 1 cols), where the +1 col is the baseline.
    fig, axes = plt.subplots(
        len(GAMMA_GRID), len(LAMBDA_GRID) + 1,
        figsize=(2.2 * (len(LAMBDA_GRID) + 1), 2.2 * len(GAMMA_GRID)),
        squeeze=False,
    )
    for i, gam in enumerate(GAMMA_GRID):
        for j, lam in enumerate(LAMBDA_GRID):
            ax = axes[i, j]
            ax.imshow(grid_results[(lam, gam, k2_val)])
            if i == 0:
                ax.set_title(f'λ={lam}', fontsize=9)
            if j == 0:
                ax.set_ylabel(f'γ={gam}', fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        # Baseline cell (rightmost column). Show it only on the middle
        # row to avoid repetition, and grey out the other baseline cells.
        ax_b = axes[i, len(LAMBDA_GRID)]
        if i == len(GAMMA_GRID) // 2:
            ax_b.imshow(blip_results[k2_val])
            ax_b.set_title('BLIP\nbaseline', fontsize=9, fontweight='bold')
        else:
            ax_b.set_facecolor('#f5f5f5')
        ax_b.set_xticks([]); ax_b.set_yticks([])

    fig.suptitle(
        f'cross_replace_steps = {k2_val}\n'
        f'instruction: {INSTRUCTION!r} | baseline tgt: {MANUAL_TARGET_PROMPT!r}',
        fontsize=10,
    )
    fig.tight_layout()
    # Render to PIL via in-memory PNG.
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    frame = Image.open(buf).convert('RGB')
    if target_size is None:
        target_size = frame.size
    else:
        frame = frame.resize(target_size)
    frames.append(frame)

# Save GIF — 1.5s per frame, infinite loop.
gif_path = OUTPUT_DIR / 'vis2c_cross_replace_cycle.gif'
frames[0].save(
    gif_path, save_all=True, append_images=frames[1:],
    duration=1500, loop=0,
)
print(f"Saved Vis #2C → {gif_path}")
# Show the first frame inline so the reader sees the layout.
display(frames[0])

In [ ]:
# 19. Save per-cell PNGs + run_metadata.json. Combined with the composed
#     figures (Vis #2A panels, Vis #2B strip comparison, Vis #2C GIF,
#     Vis #3 baseline strip) and the prompt-evolution HTML (Vis #1),
#     the output directory is a self-contained record of the run.
import json
import datetime

source_image.save(OUTPUT_DIR / 'source.png')
for (lam, gam, k2), img in grid_results.items():
    img.save(OUTPUT_DIR / f'method_lam{lam}_gam{gam}_xrep{k2}.png')
for k2, img in blip_results.items():
    img.save(OUTPUT_DIR / f'baseline_xrep{k2}.png')

# Persist PEZ-1's expensive outputs alongside this run's artifacts.
# These are also in cache/pez_source_prompts/ keyed by image+config hash,
# but copies here keep the run output directory self-contained for
# reproducibility and easy hand-off.
import torch as _torch_save
_torch_save.save(
    source_embeddings.detach().cpu(),
    OUTPUT_DIR / 'pez_1_source_embeddings.pt',
)
_torch_save.save(
    [nt.detach().cpu() for nt in null_text_per_timestep],
    OUTPUT_DIR / 'pez_1_null_text_per_timestep.pt',
)
print(f'  Saved PEZ-1 source_embeddings: shape {tuple(source_embeddings.shape)}')
print(f'  Saved PEZ-1 null_text_per_timestep: {len(null_text_per_timestep)} tensors '
      f'each shape {tuple(null_text_per_timestep[0].shape)}')

# Per-position cosine similarity between PEZ-1 and each PEZ-2 output
# — the per-position drift summary is the v1 analog of token-ID overlap.
import torch.nn.functional as _F
def _per_position_cos(src, tgt):
    a = src.squeeze(0).float()
    b = tgt.squeeze(0).float()
    return _F.cosine_similarity(a, b, dim=-1).cpu().tolist()

metadata = {
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'image_path': IMAGE_PATH,
    'image_url': IMAGE_URL,
    'instruction': INSTRUCTION,
    'manual_target_prompt': MANUAL_TARGET_PROMPT,
    'blip_caption': BLIP_SOURCE_PROMPT,
    'pez_1': {
        'embeddings_shape': list(source_embeddings.shape),
        'projected_prompt': pez_1_prompt,
    },
    'pez_2_results': {
        f'lam{lam}_gam{gam}': {
            'lambda_instruction': lam,
            'gamma_anchor': gam,
            'projected_prompt': prompt,
            'per_position_cosine_to_pez1': _per_position_cos(source_embeddings, target_emb),
        }
        for (lam, gam), (target_emb, prompt) in pez_2_results.items()
    },
    'lambda_grid': LAMBDA_GRID,
    'gamma_grid':  GAMMA_GRID,
    'knob_2_grid': list(KNOB_2_GRID),
    'model_ids': {
        'sd': edit_config.sd_model,
        'clip': CLIP_HF_ID,
        'blip': BLIP_HF_ID,
    },
    'configs': {
        'pez_1': pez_1_config.__dict__,
        'pez_2_base': {k: v for k, v in pez_2_config.__dict__.items()
                       if k not in {'lambda_instruction', 'gamma_anchor'}},
        'edit': {
            'sd_model': edit_config.sd_model,
            'ddim': edit_config.ddim.__dict__,
            'cross_attention_base': edit_config.cross_attention.__dict__,
            'alignment_method': edit_config.alignment_method,
            'alignment_threshold': edit_config.alignment_threshold,
            'mode': edit_config.mode,
        },
        'local_blend': load_local_blend().__dict__,
    },
    'output_files': {
        'source':                  'source.png',
        'pez_1_source_embeddings': 'pez_1_source_embeddings.pt',
        'pez_1_null_text':         'pez_1_null_text_per_timestep.pt',
        'method_grid_cells':  [f'method_lam{lam}_gam{gam}_xrep{k2}.png'
                               for lam in LAMBDA_GRID for gam in GAMMA_GRID for k2 in KNOB_2_GRID],
        'baseline_cells':     [f'baseline_xrep{k2}.png' for k2 in KNOB_2_GRID],
        'vis1_html':          'vis1_prompt_evolution.html',
        'vis1a_drift_summary':'vis1a_lambda_gamma_drift_summary.png',
        'vis1b_heatmap':      'vis1b_position_drift.png',
        'vis2a_panels':       [f'vis2a_method_grid_xrep{k2}.png' for k2 in KNOB_2_GRID],
        'vis2b_strip':        'vis2b_strip_comparison.png',
        'vis2c_gif':          'vis2c_cross_replace_cycle.gif',
        'vis3_baseline_row':  'vis3_baseline_row.png',
    },
}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')

n_pngs = 1 + len(grid_results) + len(blip_results) + 5  # +5 for vis1a, vis1b, 3× vis2a, vis2b
print(f'Saved {n_pngs} PNGs + 1 GIF + 1 HTML + 1 JSON to {OUTPUT_DIR}/')
print('Files:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name}')